# Sklearn Pipelines — End-to-End Preprocessing

## INFO 442 Group 5 Shuyang Zhang 320230942711

In [18]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin

titanic = pd.read_csv('titanic.csv')

### 1. Add a LogTransformer custom step that applies log1p to only the fare column before the ColumnTransformer, and show it improves cross-validated accuracy.

In [21]:
class LogTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self, columns=None):
        self.columns = columns
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        if self.columns:
            for col in self.columns:
                if col in X.columns:
                    X[col] = np.log1p(X[col].clip(lower=0))
        return X


X = titanic[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor_no_log = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), ['age', 'sibsp', 'parch', 'fare']),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), ['sex', 'embarked']),
    ('ord', SimpleImputer(strategy='most_frequent'), ['pclass'])
])

pipe_no_log = Pipeline([
    ('pre', preprocessor_no_log),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

preprocessor_with_log = ColumnTransformer([
    ('num', Pipeline([
        ('log', LogTransformer(columns=['fare'])),
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), ['age', 'sibsp', 'parch', 'fare']),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), ['sex', 'embarked']),
    ('ord', SimpleImputer(strategy='most_frequent'), ['pclass'])
])

pipe_with_log = Pipeline([
    ('pre', preprocessor_with_log),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

scores_no_log = cross_val_score(pipe_no_log, X_train, y_train, cv=5, scoring='accuracy')
scores_with_log = cross_val_score(pipe_with_log, X_train, y_train, cv=5, scoring='accuracy')

print(f"Without log transform (fare) - CV accuracy: {scores_no_log.mean():.4f} +/- {scores_no_log.std():.4f}")
print(f"With log transform (fare) - CV accuracy: {scores_with_log.mean():.4f} +/- {scores_with_log.std():.4f}")
print(f"Improvement: {scores_with_log.mean() - scores_no_log.mean():+.4f}")
print()


Without log transform (fare) - CV accuracy: 0.7922 +/- 0.0258
With log transform (fare) - CV accuracy: 0.7950 +/- 0.0271
Improvement: +0.0028



answer: Adding LogTransformer (log1p) to the fare column improved CV accuracy from 0.7922 ± 0.0258 to 0.7950 ± 0.0271, a modest improvement of +0.0028. 

### 2. Extend the Titanic pipeline to also include the deck column (extracted from the cabin number). Handle its high missingness inside the Pipeline.

In [30]:
print(f"Deck value counts:")
print(titanic['deck'].value_counts().head(10))
print(f"\nMissing deck: {titanic['deck'].isnull().sum()} rows out of {len(titanic)}")

X = titanic[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
X_with_deck = X.copy()
X_with_deck['deck'] = titanic['deck'].fillna('U')  

print(f"\nAfter filling missing with 'U':")
print(X_with_deck['deck'].value_counts().head(10))

preprocessor_with_deck = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), ['age', 'sibsp', 'parch', 'fare']),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), ['sex', 'embarked', 'deck']),
    ('ord', SimpleImputer(strategy='most_frequent'), ['pclass'])
])

pipe_with_deck = Pipeline([
    ('pre', preprocessor_with_deck),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

y = titanic['survived']
X_train_deck, X_test_deck, y_train_deck, y_test_deck = train_test_split(
    X_with_deck, y, test_size=0.2, random_state=42
)

scores_deck = cross_val_score(pipe_with_deck, X_train_deck, y_train_deck, cv=5, scoring='accuracy')
print(f"\nCV accuracy with deck column: {scores_deck.mean():.4f} +/- {scores_deck.std():.4f}")

X_no_deck = X.copy()
preprocessor_no_deck = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), ['age', 'sibsp', 'parch', 'fare']),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), ['sex', 'embarked']),
    ('ord', SimpleImputer(strategy='most_frequent'), ['pclass'])
])

pipe_no_deck = Pipeline([
    ('pre', preprocessor_no_deck),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

X_train_no_deck, X_test_no_deck, _, _ = train_test_split(
    X_no_deck, y, test_size=0.2, random_state=42
)

scores_no_deck = cross_val_score(pipe_no_deck, X_train_no_deck, y_train_deck, cv=5, scoring='accuracy')

print(f"\nCV accuracy WITHOUT deck: {scores_no_deck.mean():.4f} +/- {scores_no_deck.std():.4f}")
print(f"CV accuracy WITH deck: {scores_deck.mean():.4f} +/- {scores_deck.std():.4f}")
print(f"Improvement from deck: {scores_deck.mean() - scores_no_deck.mean():+.4f}")

Deck value counts:
deck
C    59
B    47
D    33
E    32
A    15
F    13
G     4
Name: count, dtype: int64

Missing deck: 688 rows out of 891

After filling missing with 'U':
deck
U    688
C     59
B     47
D     33
E     32
A     15
F     13
G      4
Name: count, dtype: int64

CV accuracy with deck column: 0.7964 +/- 0.0355

CV accuracy WITHOUT deck: 0.7922 +/- 0.0258
CV accuracy WITH deck: 0.7964 +/- 0.0355
Improvement from deck: +0.0042


### 3. Use GridSearchCV to tune the imputation strategy (median vs most_frequent) for the age column alongside RandomForest hyperparameters. Use Pipeline parameter notation.

In [33]:
from sklearn.model_selection import GridSearchCV

X = titanic[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor_grid = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer()), 
        ('scale', StandardScaler()),
    ]), ['age', 'sibsp', 'parch', 'fare']),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), ['sex', 'embarked']),
    ('ord', SimpleImputer(strategy='most_frequent'), ['pclass'])
])

pipe_grid = Pipeline([
    ('pre', preprocessor_grid),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid = {
    'pre__num__impute__strategy': ['median', 'mean'],
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 8, None]
}

print("Parameter grid:")
for key, values in param_grid.items():
    print(f"  {key}: {values}")
print()

print("Running GridSearchCV (this may take a moment)...")
grid_search = GridSearchCV(
    pipe_grid, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")
print(f"Best imputation strategy for age: {grid_search.best_params_['pre__num__impute__strategy']}")

test_accuracy = grid_search.score(X_test, y_test)
print(f"Test accuracy: {test_accuracy:.4f}")
print()

Parameter grid:
  pre__num__impute__strategy: ['median', 'mean']
  model__n_estimators: [100, 200]
  model__max_depth: [5, 8, None]

Running GridSearchCV (this may take a moment)...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best parameters: {'model__max_depth': 5, 'model__n_estimators': 100, 'pre__num__impute__strategy': 'median'}
Best CV accuracy: 0.8272
Best imputation strategy for age: median
Test accuracy: 0.8045



### 4. Demonstrate that fitting the scaler outside the Pipeline before cross_val_score inflates the CV score — show the numerical difference. 

In [50]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

X, y = make_classification(n_samples=200, n_features=10, random_state=42)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_test = X_test * 2

X_combined = np.vstack([X_train, X_test])
y_combined = np.hstack([y_train, y_test])

scaler = StandardScaler()
X_scaled_all = scaler.fit_transform(X_combined)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = SVC(kernel='linear', random_state=42)

scores_leaked = []
fold = 1
for train_idx, val_idx in kf.split(X_scaled_all):
    X_tr, X_val = X_scaled_all[train_idx], X_scaled_all[val_idx]
    y_tr, y_val = y_combined[train_idx], y_combined[val_idx]
    model.fit(X_tr, y_tr)
    score = model.score(X_val, y_val)
    scores_leaked.append(score)
    print(f"  Fold {fold}: {score:.4f}")
    fold += 1
print(f"  Mean CV accuracy (leaked): {np.mean(scores_leaked):.4f} +/- {np.std(scores_leaked):.4f}\n")

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', SVC(kernel='linear', random_state=42))
])

scores_correct = cross_val_score(pipe, X_combined, y_combined, cv=kf)
print(f"  CV accuracy: {scores_correct.mean():.4f} +/- {scores_correct.std():.4f}\n")

print(f"Pipeline:     {scores_correct.mean():.4f} +/- {scores_correct.std():.4f}")
print(f"Scale first:   {np.mean(scores_leaked):.4f} +/- {np.std(scores_leaked):.4f}")

  Fold 1: 0.8500
  Fold 2: 0.8500
  Fold 3: 0.8000
  Fold 4: 0.8250
  Fold 5: 0.8250
  Mean CV accuracy (leaked): 0.8300 +/- 0.0187

  CV accuracy: 0.8250 +/- 0.0274

Pipeline:     0.8250 +/- 0.0274
Scale first:   0.8300 +/- 0.0187
